# EDA 004.01 — Univariate Analysis

**Exploring one variable at a time**

## Learning Objectives
By the end of this notebook you will be able to:
- Choose the right visualization for numerical vs categorical data
- Create and interpret histograms, KDE plots, box plots, and violin plots
- Identify skewness, outliers, and multimodality from a single chart
- Use count plots and bar charts for categorical variables

---

## What Is Univariate Analysis?

Univariate analysis examines **one variable at a time** to understand its distribution,
central tendency, spread, and shape. It is always the **first step** in any EDA.

| Variable type | Best plot(s) |
|---|---|
| Numerical (continuous) | Histogram + KDE, Box plot, Violin plot |
| Numerical (discrete) | Bar chart, Histogram |
| Categorical (nominal/ordinal) | Count plot, Sorted bar chart |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

df = sns.load_dataset('titanic')
print(f'Shape: {df.shape}')
df[['age', 'fare', 'pclass', 'sex', 'embarked']].head()

---
## 1 — Histogram

**References:** [seaborn.histplot](https://seaborn.pydata.org/generated/seaborn.histplot.html) | [matplotlib.hist](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html)

A histogram divides the range into **bins** and counts observations per bin.

- **Too few bins** → over-smoothed; hides detail
- **Too many bins** → noisy; hard to read the shape
- Sturges' rule for bin count: $k = \lceil \log_2 n \rceil + 1$

Add `kde=True` to overlay a smooth density estimate.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

age = df['age'].dropna()

sns.histplot(age, bins=5, ax=axes[0], color='steelblue')
axes[0].set_title('Too few bins (5) — loses shape')

sns.histplot(age, bins=30, kde=True, ax=axes[1], color='steelblue')
axes[1].set_title('Good bins (30) + KDE overlay')

sns.histplot(age, bins=100, ax=axes[2], color='steelblue')
axes[2].set_title('Too many bins (100) — noisy')

plt.suptitle('Effect of Bin Width — Titanic Passenger Age', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Sturges rule suggests: {int(np.ceil(np.log2(len(age)))) + 1} bins')

---
## 2 — KDE Plot (Kernel Density Estimate)

**Reference:** [seaborn.kdeplot](https://seaborn.pydata.org/generated/seaborn.kdeplot.html)

KDE is a **smoothed, continuous** version of the histogram — no binning required.

- `bw_adjust` controls smoothing bandwidth (higher = smoother)
- Excellent for **overlaying multiple groups** to compare distributions
- The area under the curve always integrates to 1 (it's a probability density)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bandwidth comparison
for bw, ls in [(0.3, '--'), (1.0, '-'), (2.0, ':')]:
    sns.kdeplot(age, bw_adjust=bw, ax=axes[0], label=f'bw_adjust={bw}', linestyle=ls)
axes[0].set_title('Effect of Bandwidth')
axes[0].legend()

# Groups comparison
data_survived = df.dropna(subset=['age'])
sns.kdeplot(data=data_survived, x='age', hue='survived',
            fill=True, alpha=0.35, ax=axes[1],
            palette={0: 'coral', 1: 'steelblue'})
axes[1].set_title('Age Distribution: Survived vs Not Survived')

plt.tight_layout()
plt.show()

---
## 3 — Box Plot

**Reference:** [seaborn.boxplot](https://seaborn.pydata.org/generated/seaborn.boxplot.html)

Box plots visualise the **five-number summary**: min whisker, Q1, median, Q3, max whisker.

```
                  |-------[===|===]-------|   o   o
               lower    Q1  med  Q3    upper  outliers
               whisker                 whisker
               (Q1 - 1.5*IQR)         (Q3 + 1.5*IQR)
```

| Feature | What it tells you |
|---|---|
| Median line | Central tendency |
| Box width | IQR = middle 50% spread |
| Whisker length | Overall spread (excl. outliers) |
| Points beyond whiskers | Outliers |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Horizontal box — shows skew + outliers clearly
sns.boxplot(x=df['fare'], ax=axes[0], color='lightcoral')
axes[0].set_title('Fare Distribution — right-skewed with outliers')

# Grouped box — compare distributions across categories
sns.boxplot(data=df, x='pclass', y='age', ax=axes[1],
            palette='Set2', order=[1, 2, 3])
axes[1].set_title('Age by Passenger Class')
axes[1].set_xlabel('Passenger Class (1=First, 3=Third)')

plt.tight_layout()
plt.show()

---
## 4 — Violin Plot

**Reference:** [seaborn.violinplot](https://seaborn.pydata.org/generated/seaborn.violinplot.html)

A violin plot = **box plot + KDE mirrored on both sides**.
Width at each y-value represents the density of observations at that value.

| Feature | Box plot | Violin plot |
|---|---|---|
| Shows quartiles | Yes | Yes (inner='box') |
| Shows distribution shape | No | Yes |
| Reveals bimodality | No | Yes |
| Readability | High | Medium |

> Use **box plots** for reports; use **violin plots** during exploration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.violinplot(data=df, x='pclass', y='age', ax=axes[0],
               palette='Set2', order=[1, 2, 3], inner='box')
axes[0].set_title('Violin: Age by Class (inner=box)')
axes[0].set_xlabel('Passenger Class')

# Split violin — compare two groups within each category in one plot
df_clean = df.dropna(subset=['age'])
sns.violinplot(data=df_clean, x='pclass', y='age', hue='sex',
               split=True, ax=axes[1], palette='Set1',
               order=[1, 2, 3], inner='quart')
axes[1].set_title('Violin: Age by Class × Sex (split violin)')
axes[1].set_xlabel('Passenger Class')

plt.tight_layout()
plt.show()

---
## 5 — Count Plot (Categorical Variables)

**Reference:** [seaborn.countplot](https://seaborn.pydata.org/generated/seaborn.countplot.html)

For categorical variables, a count plot shows the frequency of each category.

Tips:
- **Sort** by frequency for unordered categories
- Add **percentage labels** for easier interpretation
- Avoid pie charts — humans are poor at comparing angles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Embarkation port — sorted by count
order_emb = df['embarked'].value_counts().index
sns.countplot(data=df, x='embarked', order=order_emb, ax=axes[0], palette='Set2')
axes[0].set_title('Embarkation Port')

# Passenger class
sns.countplot(data=df, x='pclass', ax=axes[1], palette='Set3', order=[1, 2, 3])
axes[1].set_title('Passenger Class')

# Sex — with percentage labels
order_sex = df['sex'].value_counts().index
sns.countplot(data=df, x='sex', order=order_sex, ax=axes[2], palette='pastel')
total = len(df)
for p in axes[2].patches:
    pct = f'{100 * p.get_height() / total:.1f}%'
    axes[2].annotate(pct,
                     (p.get_x() + p.get_width() / 2, p.get_height() + 4),
                     ha='center', fontsize=11, fontweight='bold')
axes[2].set_title('Sex (with % labels)')

plt.tight_layout()
plt.show()

---
## 6 — Choosing the Right Plot

| Question | Best plot |
|---|---|
| How is this numeric variable distributed? | Histogram + KDE |
| Are there outliers? Where are the quartiles? | Box plot |
| Is the distribution bimodal or has unusual shape? | Violin plot |
| How many records per category? | Count plot |
| Compare distributions across groups? | Grouped box / split violin / overlaid KDE |

---
## Key Takeaways

- Always start EDA with univariate analysis — understand each variable before combining them
- Histograms reveal **shape**, KDE reveals **density** (no binning needed)
- Box plots efficiently show **quartiles and outliers**
- Violin plots are richer — combine box summary with density shape
- Count plots are the primary tool for categorical variables
- Look for: skewness, multimodality, outliers, data range, and unexpected values

---
## Further Reading

- [Kaggle — Data Visualization Course](https://www.kaggle.com/learn/data-visualization) (seaborn-based)
- [Seaborn Tutorial](https://seaborn.pydata.org/tutorial.html)
- [Fundamentals of Data Visualization (free online)](https://clauswilke.com/dataviz/) — Claus Wilke
- [The Visual Display of Quantitative Information](https://www.edwardtufte.com/tufte/books_vdqi) — Edward Tufte
- [Matplotlib Gallery](https://matplotlib.org/stable/gallery/index.html)